# Indian Traffic Sign Recognition (TSR) - Cloud GPU Training Pipeline
### Direct Cloud Dataset Download & YOLOv8n Training on Google Colab (Tesla T4 / L4)

This notebook provides a complete zero-local-storage workflow:
1. **Check GPU acceleration** (Tesla T4 / L4)
2. **Download real Indian Road Traffic Sign datasets directly in Colab** (using Kaggle / Roboflow / Direct Curl)
3. **Configure `dataset.yaml`**
4. **Train YOLOv8n detector & classifier** (100 epochs, 640x640, AdamW)
5. **Evaluate metrics** (mAP@50, mAP@50-95, Precision, Recall, Confusion Matrix)
6. **Export `best.pt`** directly to Google Drive or download to your local `models/` directory

In [ ]:
# Step 1: Verify Colab GPU acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics YOLO and high-speed data utilities
!pip install --upgrade ultralytics opencv-python matplotlib kagglehub roboflow

## Step 3: Download Real Indian Traffic Sign Dataset Directly in Colab
Choose **Method A** (KaggleHub - automated), **Method B** (Kaggle CLI), or **Method C** (Roboflow Universe).

In [ ]:
# METHOD A (Recommended - Instant Download via kagglehub):
# Downloads real Indian road traffic signs directly into Colab SSD in seconds
import kagglehub
import os, shutil

print("Downloading real Indian road traffic signs dataset from Kaggle...")
# Real Indian Roads Traffic Sign Dataset
download_path = kagglehub.dataset_download("kaustubhrastogi/traffic-signs-dataset-in-indian-roads")
print("Downloaded to:", download_path)

# Inspect contents and prepare /content/dataset
os.makedirs("/content/dataset", exist_ok=True)
for item in os.listdir(download_path):
    s = os.path.join(download_path, item)
    d = os.path.join("/content/dataset", item)
    if os.path.isdir(s):
        if os.path.exists(d):
            shutil.rmtree(d)
        shutil.copytree(s, d)
    else:
        shutil.copy2(s, d)

print("Dataset prepared in /content/dataset:", os.listdir("/content/dataset"))

In [ ]:
# METHOD B (Alternative: Roboflow Universe Indian Signs in YOLOv8 format):
# If using a Roboflow dataset, uncomment and enter your free API key:
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")
# project = rf.workspace("indian-traffic-signs").project("indian-traffic-signs-dataset")
# version = project.version(1)
# dataset = version.download("yolov8")

In [ ]:
# Step 4: Verify or Generate dataset.yaml configuration
# Check if dataset already has a data.yaml, otherwise write our standard Indian TSR config:
import os

existing_yaml = [f for f in os.listdir("/content/dataset") if f.endswith(".yaml")]
if existing_yaml:
    dataset_yaml_path = os.path.join("/content/dataset", existing_yaml[0])
    print(f"Found existing dataset YAML: {dataset_yaml_path}")
    with open(dataset_yaml_path, 'r') as f:
        print(f.read())
else:
    dataset_yaml_path = "/content/dataset.yaml"
    yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 15
names:
  0: Stop
  1: No Entry
  2: Speed Limit 20
  3: Speed Limit 40
  4: Speed Limit 60
  5: Speed Limit 80
  6: Give Way
  7: School Ahead
  8: Pedestrian Crossing
  9: Left Turn
  10: Right Turn
  11: U-Turn
  12: Horn Prohibited
  13: Parking
  14: Dangerous Curve
"""
    with open(dataset_yaml_path, "w") as f:
        f.write(yaml_content.strip())
    print(f"Created default configuration at: {dataset_yaml_path}")

In [ ]:
# Step 5: Mount Google Drive (to save models and persistent weights)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 6: Train YOLOv8n on Cloud GPU (Tesla T4)
from ultralytics import YOLO

# Load pretrained model weights
model = YOLO("yolov8n.pt")

# Train for 100 epochs (optimized for speed & accuracy)
results = model.train(
    data=dataset_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    device=0,
    project="tsr_colab_run",
    name="yolov8n_indian_tsr",
    save=True,
    plots=True
)

In [ ]:
# Step 7: Evaluate Model Metrics on Validation Set
metrics = model.val()
print(f"Validation Results on Real Indian Road Test Data:")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Step 8: Visualize Results and Confusion Matrix
from IPython.display import Image, display
import glob

results_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/results.png")
if results_png:
    print("--- Training Loss & Metric Curves ---")
    display(Image(results_png[0]))

cm_png = glob.glob("/content/tsr_colab_run/yolov8n_indian_tsr/confusion_matrix.png")
if cm_png:
    print("--- Confusion Matrix ---")
    display(Image(cm_png[0]))

In [ ]:
# Step 9: Export best.pt to Google Drive & download to your laptop
import shutil
from google.colab import files

best_weights = "/content/tsr_colab_run/yolov8n_indian_tsr/weights/best.pt"
if os.path.exists(best_weights):
    # Save backup to Google Drive
    os.makedirs("/content/drive/MyDrive/TSR_Models", exist_ok=True)
    shutil.copy(best_weights, "/content/drive/MyDrive/TSR_Models/best.pt")
    print("Successfully saved best weights to Google Drive: /content/drive/MyDrive/TSR_Models/best.pt")
    
    # Download directly to your laptop's 'models/' folder
    files.download(best_weights)
else:
    print("best.pt was not found. Please review the training logs above.")